In [ ]:
library(dplyr)

simulate_data <- function(true_betas, is_binary = NULL, noise_strength = 1, n = 100) {
  p <- length(true_betas) - 1  # number of predictors (excluding intercept)

  if (is.null(is_binary)) {
    is_binary <- rep(0, p)
  }

  if (length(is_binary) != p) {
    stop("Length of is_binary must match the number of predictors (length(true_betas) - 1).")
  }

  # Simulate predictors
  predictors <- matrix(nrow = n, ncol = p)
  for (j in 1:p) {
    if (is_binary[j] == 1) {
      predictors[, j] <- rbinom(n, size = 1, prob = 0.5)
    } else {
      predictors[, j] <- rnorm(n)
    }
  }

  colnames(predictors) <- paste0("x", 1:p)
  X <- as.data.frame(predictors)

  # Linear predictor: b0 + X %*% b
  eta <- true_betas[1] + as.matrix(X) %*% true_betas[-1]

  # Add noise to generate the outcome
  y <- as.vector(eta + rnorm(n, mean = 0, sd = noise_strength))

  # Return full data frame
  return(cbind(X, y = y))
}


set.seed(123)

## Nurses linear

In [ ]:
set.seed(123)

df <- read.csv("data/summarized/params.Nurses.csv")
covs <- df[df["Method"] == "Combined", c("Covariate", "Estimate")]
covs
is_binary <- c(1, 0, 0, 1)
stopifnot(covs$Covariate[[1]] == "(Intercept)")
true_betas <- covs$Estimate
true_betas <- true_betas[1:length(true_betas)-1]

sigma <- as.numeric(covs[covs["Covariate"] == "sigma2"][[2]])
n <- 1000 - 80
hospitals <- 23
df <- simulate_data(true_betas, is_binary, noise_strength = sigma, n = n)
df$hospital <- sample(seq(hospitals), n, replace = TRUE)

# add outlier hospital (intercept shift)

true_betas_o <- true_betas
true_betas_o[[1]] <- true_betas_o[[1]] + 1
sigma <- sigma
n <- 40
dfo <- simulate_data(true_betas_o, is_binary, noise_strength = sigma, n = n)
dfo$hospital <- hospitals + 1

# add outlier hospital (error variance shift)

sigma <- 2*sigma
n <- 40
dfo2 <- simulate_data(true_betas, is_binary, noise_strength = sigma, n = n)
dfo2$hospital <- hospitals + 2

df2 <- rbind(df, dfo, dfo2)
df2 <- df2 |>
  rename(
    gender = x1,
    age = x2,
    experien = x3,
    wardtype = x4,
    stress = y
  )

df <- df2[df2["hospital"] <= hospitals, ]
write.csv(df, "data/raw/sim-nurses-1.csv", row.names = TRUE)
write.csv(df2, "data/raw/sim-nurses-1-out.csv", row.names = TRUE)
table(df2$hospital)
tail(df2)

## Test data sets

In [ ]:
set.seed(123)

true_betas <- c(0, 0.5, -2, 1, -1, 3)
sigma <- 3
n <- 1000
hospitals <- 25
df <- simulate_data(true_betas, noise_strength = sigma, n = n)
df$hospital <- sample(seq(hospitals), n, replace = TRUE)
table(df$hospital)
head(df)
write.csv(df, "data/raw/sim-linear-1.csv", row.names = TRUE)

# save truth
cnames <- head(colnames(df), -2)
truth <- setNames(true_betas[2:(length(true_betas))], cnames)
truth$sigma2 <- sigma*sigma
truth$Intercept <- true_betas[1]
write.csv(truth, "data/raw/sim-linear-1.truth.csv", row.names=FALSE)

In [ ]:
# add outlier hospitals

true_betas <- 2 + c(0, 0.5, -2, 1, -1, 3)
sigma <- 3
n <- 40
hospitals <- 1
dfo <- simulate_data(true_betas, noise_strength = sigma, n = n)
dfo$hospital <- sample(seq(hospitals), n, replace = TRUE)
dfo$hospital <- 25 + dfo$hospital

# save truth
cnames <- head(colnames(dfo), -2)
trutho <- setNames(true_betas[2:(length(true_betas))], cnames)
trutho$sigma2 <- sigma*sigma
trutho$Intercept <- true_betas[1]

In [ ]:
#df2 <- rbind(df[df$hospital < 7, ], dfo)
df2 <- rbind(df, dfo)
write.csv(df2, "data/raw/sim-linear-1-out.csv", row.names = TRUE)